# Lab Day 2 — DeepWeeds on Kaggle / Colab

Notebook tái lập EDA, so sánh backbone, ablation công thức, suy luận và vòng chung kết. Mọi lựa chọn cấu hình dùng fold-0 validation; test chỉ dùng ở vòng chung kết, một lượt cho mỗi seed.

Trên Kaggle, bật **GPU Accelerator** và **Internet** trong Settings rồi chạy các ô theo thứ tự. Notebook chứa snapshot `eval.py` + `starter/*.py` và tự tải DeepWeeds; không cần upload thêm mã nguồn hay dataset. Bước suy luận chạy trên checkpoint đã chọn, không cập nhật trọng số.


## 0. Cài thư viện


In [ ]:
%pip -q install timm fvcore openpyxl scikit-learn


## 1. Mã nguồn và môi trường

Cell kế tiếp giải nén snapshot mã nguồn được nhúng trong notebook vào thư mục làm việc. Dataset và checkpoint không nằm trong snapshot.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import base64, json, os, platform, sys, zipfile

from IPython.display import Image, FileLink, display

SOURCE_BUNDLE_BASE64 = """..."""
WORK_ROOT = Path("/kaggle/working" if Path("/kaggle/working").is_dir() else "/content")
ROOT = WORK_ROOT / "K4-Track4-Day2-Deeplearning-Advance"
ROOT.mkdir(parents=True, exist_ok=True)
with ZipFile(BytesIO(base64.b64decode(SOURCE_BUNDLE_BASE64))) as archive:
    archive.extractall(ROOT)
assert (ROOT / "eval.py").is_file() and (ROOT / "starter" / "lab_workflow.py").is_file(), ROOT
os.chdir(ROOT)
CODE_DIR = ROOT / "starter"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(CODE_DIR))

%pip -q install timm fvcore openpyxl scikit-learn
import numpy as np, pandas as pd, torch, timm
print("Python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
assert torch.cuda.is_available(), "Bật GPU Accelerator trong Kaggle Settings trước khi chạy."
print("GPU:", torch.cuda.get_device_name(0))
OUTPUT_DIR = WORK_ROOT / "lab-output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ENVIRONMENT = {"python": platform.python_version(), "torch": torch.__version__,
               "timm": timm.__version__, "numpy": np.__version__, "pandas": pd.__version__,
               "gpu": torch.cuda.get_device_name(0)}
(OUTPUT_DIR / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2), encoding="utf-8")


## 2. Tải DeepWeeds và giữ nguyên fold 0

Cell kế tiếp cần Internet để tải ảnh từ Zenodo và CSV fold gốc từ GitHub.


In [ ]:
import hashlib, shutil, urllib.request

DATA_DIR = ROOT / 'data'
LABELS_DIR = DATA_DIR / 'labels'
DATA_DIR.mkdir(exist_ok=True)
LABELS_DIR.mkdir(exist_ok=True)
archive_path = DATA_DIR / 'images.zip'
if not archive_path.exists():
    url = 'https://zenodo.org/records/7939060/files/images.zip?download=1'
    urllib.request.urlretrieve(url, archive_path)
hasher = hashlib.md5()
with archive_path.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1 << 20), b''):
        hasher.update(chunk)
digest = hasher.hexdigest()
assert digest == 'b7b30f96d466fba86016aa5a26606e0f', f'MD5 không khớp: {digest}'
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(DATA_DIR)
base_url = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels'
for name in ('labels', 'train_subset0', 'val_subset0', 'test_subset0'):
    target = LABELS_DIR / f'{name}.csv'
    if not target.exists():
        urllib.request.urlretrieve(f'{base_url}/{name}.csv', target)

from dataset import discover_images_dir
IMAGES_DIR = discover_images_dir(DATA_DIR)
print('images.zip MD5:', digest)
print('Images:', IMAGES_DIR)
print('JPG count:', sum(1 for p in IMAGES_DIR.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg'}))


## 3. Bước 0 — EDA, chia dữ liệu và pipeline

Kiểm tra giao rỗng, tổng 17.509 ảnh, mọi file ảnh và phân bố từng lớp. Ví dụ augmentation lấy từ train. Smoke test kiểm tra loss ban đầu và khả năng overfit một batch nhỏ.


In [ ]:
from lab_workflow import prepare_data_and_eda, pipeline_smoke
split_report = prepare_data_and_eda(IMAGES_DIR, LABELS_DIR, OUTPUT_DIR)
display(pd.read_csv(OUTPUT_DIR / 'eda_class_counts.csv'))
display(Image(filename=str(OUTPUT_DIR / 'eda_class_distribution.png')))
display(Image(filename=str(OUTPUT_DIR / 'eda_train_examples.png')))
smoke = pipeline_smoke(IMAGES_DIR, LABELS_DIR, seed=0, output_dir=OUTPUT_DIR)
print(smoke)


## 4. Bước 1–3 — Backbone, công thức huấn luyện và suy luận

Notebook dùng năm backbone và các ablation đã khai báo trong mã. Bước suy luận I00–I08 chạy trên mô hình/checkpoint đã huấn luyện, chọn cấu hình chỉ bằng validation và đo riêng batch 1 và batch 32.


In [ ]:
from lab_workflow import run_backbone_and_recipe_study
EPOCHS = 10
BATCH_SIZE = 32
study = run_backbone_and_recipe_study(IMAGES_DIR, LABELS_DIR, OUTPUT_DIR,
                                      epochs=EPOCHS, batch_size=BATCH_SIZE, num_workers=2)
display(pd.DataFrame([{k:v for k,v in r.items() if k not in {'split','test_metrics'}}
                      for r in study['backbones']]))
display(pd.DataFrame([{k:v for k,v in r.items() if k not in {'split','test_metrics'}}
                      for r in study['training']]))
display(pd.DataFrame(study['inference']['rows']))
print('Selected recipe:', study['selected_recipe']['exp_id'])
print('Selected inference:', study['inference']['inference_exp_id'])


## 5. Bước 4–5 — Chung kết, chấm và sản phẩm

Chỉ chạy sau khi bước trước hoàn tất. Huấn luyện lại F01 và T00 với cùng ba seed; mỗi test prediction được ghi đúng một lần. Cuối cùng chạy `eval.py score` và `eval.py grade` qua helper, rồi tạo đủ sheet trong `results.xlsx`.


In [ ]:
from lab_workflow import run_final_and_score, export_submission
final_summary = run_final_and_score(study, IMAGES_DIR, LABELS_DIR, OUTPUT_DIR,
                                    seeds=(0,1,2), num_workers=2)
print(final_summary)

# Điền đúng dạng MSSV_hoten_khong_dau sau khi nhận biến STUDENT_SLUG.
STUDENT_SLUG = "student_submission"  # Thay bằng MSSV_ten_khong_dau khi nộp.
submission_dir = export_submission(OUTPUT_DIR, STUDENT_SLUG, environment=ENVIRONMENT)
print('Submission folder:', submission_dir)


## Tải kết quả về máy

Kaggle lưu gói nộp trong `/kaggle/working`; link tải hiện ở cell dưới. Gói không chứa ảnh DeepWeeds hoặc checkpoint.


In [ ]:
import shutil
submission_zip = shutil.make_archive(str(WORK_ROOT / STUDENT_SLUG), "zip",
                                     root_dir=submission_dir.parent, base_dir=submission_dir.name)
print("Submission archive:", submission_zip)
display(FileLink(submission_zip))
